# 12 Canopy Boosting

Model family: `boosting`

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import importlib

# Resolve project root in local or Kaggle runtime.
if Path("/kaggle/working").exists():
    ROOT = Path("/kaggle/working")
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

if (ROOT / "src").exists():
    SRC = ROOT / "src"
else:
    SRC = Path.cwd() / "src"
if str(SRC) not in sys.path and SRC.exists():
    sys.path.append(str(SRC))

if importlib.util.find_spec("laspy") is None:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "laspy[lazrs]"])

from lidar_roraima.config import ProjectConfig
cfg = ProjectConfig.from_root(ROOT)
cfg.ensure_dirs()

# Kaggle dataset fallback path.
RAW_DATA_DIR = cfg.raw_data_dir
for candidate in [
    Path("/kaggle/input/lidar-roraima-parime-research/lidar_data"),
    Path("/kaggle/input/lidar-roraima-parime-research"),
]:
    if candidate.exists():
        RAW_DATA_DIR = candidate
        break

print("ROOT:", ROOT)
print("RAW_DATA_DIR:", RAW_DATA_DIR)
TRAIN_MAX_ROWS = None  # Set integer for constrained runtime, e.g. 1200000
cfg


In [ ]:
from lidar_roraima.features import load_feature_tables
from lidar_roraima.models import train_canopy_model
from lidar_roraima.registry import append_model_result

features = load_feature_tables(cfg.features_dir)
result = train_canopy_model(features_df=features, model_family="boosting", output_dir=cfg.models_dir, seed=cfg.random_seed, n_splits=5, max_rows=TRAIN_MAX_ROWS)
registry = append_model_result(result, cfg.models_dir / "model_results.csv")
result


In [ ]:
pd.read_csv(cfg.models_dir / "model_results.csv").tail(10)